# Module 1: Validation & Go/No-Go Checklist

## Overview

This notebook validates that your LangSmith deployment is healthy and ready for use. This checklist becomes your **baseline reference** for future troubleshooting.

### What We'll Validate

1. ✅ Pod readiness (all pods running)
2. ✅ License key validation (properly configured)
3. ✅ PVC binding (storage provisioned)
4. ✅ External services connectivity (PostgreSQL, Redis, blob storage)
5. ✅ Ingress provisioning (load balancer created)
6. ✅ Endpoint reachability (services accessible)
7. ✅ Basic UI availability (web interface works)
8. ✅ Basic functional test (optional trace submission)

### Why This Matters

Most issues are caught here, before real users onboard. This validation ensures you're on a **supported path**.

**Estimated time:** 20-30 minutes


In [ ]:
# Bootstrap environment
import sys
from pathlib import Path

# Add notebooks directory to path so we can import shared as a package
# Find the notebooks directory by looking for the shared folder
possible_paths = [
    Path.cwd().parent,  # If cwd is module-1, go up one level to notebooks
    Path.cwd(),  # If cwd is already notebooks
    Path.cwd() / "notebooks",  # If cwd is workspace root
]

notebooks_path = None
for path in possible_paths:
    if path and (path / "shared" / "_bootstrap.py").exists():
        notebooks_path = path
        break

if not notebooks_path:
    notebooks_path = Path.cwd() / "notebooks"
    if not (notebooks_path / "shared" / "_bootstrap.py").exists():
        raise RuntimeError(f"Could not find notebooks/shared directory. Current dir: {Path.cwd()}")

# Add notebooks directory to path so 'shared' can be imported as a package
if str(notebooks_path) not in sys.path:
    sys.path.insert(0, str(notebooks_path))

from shared._bootstrap import bootstrap

# Run bootstrap
bootstrap_info = bootstrap()
artifacts_dir = Path(bootstrap_info['artifacts_dir'])
print(f"\nArtifacts directory: {artifacts_dir}")


## Setting Up Cluster Access

Ensure kubectl is configured for the Kubernetes cluster.


In [ ]:
import os
from shared._validation import require_env, ok, warn
from shared._cloud_helpers import (
    get_cloud_provider,
    get_region,
    configure_kubectl,
)
from shared._shell import run

provider = get_cloud_provider()

# Cloud-specific region variable
if provider == "aws":
    region_var = "AWS_REGION"
elif provider == "azure":
    region_var = "AZURE_LOCATION"
else:
    region_var = "AWS_REGION"  # Default for backward compatibility

# Get configuration
config = require_env("CLUSTER_NAME", region_var, "NAMESPACE")
cluster_name = config["CLUSTER_NAME"]
region = get_region()
namespace = config["NAMESPACE"]

# Configure kubectl
print("### Configuring kubectl\n")
configure_kubectl(cluster_name, region)
ok("kubectl configured")

# Test cluster access
result = run(["kubectl", "cluster-info"], check=True, stream=False)
print(result.stdout)


## 1. Pod Readiness Check

**Critical:** All pods must be in `Running` state with `Ready` status. This is the foundation of a healthy deployment.


In [ ]:
from shared._k8s_helpers import get_pods, wait_for_deployments_ready, require_namespace
from shared._validation import warn
from shared._shell import run
import json

# Ensure namespace exists
require_namespace(namespace)

# Wait for deployments to be ready (with timeout)
print("### Waiting for Deployments to be Ready\n")
print("This may take a few minutes if pods are still starting...\n")

try:
    wait_for_deployments_ready(namespace, timeout="10m")
except Exception as e:
    print(f"⚠️  Timeout or error waiting for deployments: {e}")
    print("💡 Some pods may still be starting. Continuing with status check...")

# Get pod status
print("\n### Pod Status\n")
pods_output = get_pods(namespace)
print(pods_output)

# Parse pod status
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
pods_data = json.loads(result.stdout)

# Analyze pod status
running = 0
pending = 0
failed = 0
ready = 0
total = len(pods_data.get("items", []))

for pod in pods_data.get("items", []):
    status = pod.get("status", {})
    phase = status.get("phase", "Unknown")
    conditions = status.get("conditions", [])
    
    if phase == "Running":
        running += 1
        # Check ready condition
        for cond in conditions:
            if cond.get("type") == "Ready" and cond.get("status") == "True":
                ready += 1
                break
    elif phase == "Pending":
        pending += 1
    elif phase == "Failed":
        failed += 1

print(f"\n### Pod Summary")
print(f"Total pods: {total}")
print(f"Running: {running}")
print(f"Ready: {ready}")
print(f"Pending: {pending}")
print(f"Failed: {failed}")

if ready == total and total > 0:
    ok(f"All {total} pods are ready")
elif running == total and total > 0:
    warn(f"All pods running but {total - ready} not ready yet")
else:
    warn(f"Pod status: {running}/{total} running, {ready}/{total} ready")
    if pending > 0:
        print("💡 Some pods are still pending. Check events for issues:")
        run(["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"], check=False, stream=True)


## 1.5. License Key Validation

**Critical:** Verify that the LangSmith license key is properly configured and valid. License issues will prevent the system from functioning correctly.


In [ ]:
# Check license key secret
print("### License Key Validation\n")

# Check if license secret exists
result = run(
    ["kubectl", "get", "secret", "langsmith-license", "-n", namespace, "-o", "json"],
    check=False,
    stream=False
)

if result.returncode == 0:
    ok("License key secret exists")
    
    # Try to check if license key is set (without revealing it)
    secret_data = json.loads(result.stdout)
    if "data" in secret_data and "license-key" in secret_data["data"]:
        ok("License key is present in secret")
    else:
        warn("License key secret exists but 'license-key' field not found")
        print("💡 Secret may use a different key name")
else:
    warn("License key secret not found")
    print("💡 License secret 'langsmith-license' should exist in namespace")
    print("   Check that you created the secret during Helm installation")

# Check pod logs for license-related errors
print("\n### Checking Pod Logs for License Errors\n")

# Get all pods in namespace
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    pod_names = result.stdout.strip().split()
    license_errors_found = False
    
    # Check logs from a few key pods (limit to first 3 to avoid too much output)
    key_pods = [p for p in pod_names if any(keyword in p.lower() for keyword in ["api", "server", "backend"])][:3]
    if not key_pods:
        key_pods = pod_names[:3]  # Fallback to first 3 pods
    
    for pod_name in key_pods:
        try:
            # Get recent logs (last 50 lines)
            log_result = run(
                ["kubectl", "logs", pod_name, "-n", namespace, "--tail=50"],
                check=False,
                stream=False
            )
            
            if log_result.returncode == 0:
                logs = log_result.stdout.lower()
                # Look for common license-related error patterns
                license_error_patterns = [
                    "license",
                    "unauthorized",
                    "invalid license",
                    "license expired",
                    "license key",
                    "beacon.langchain.com",
                ]
                
                for pattern in license_error_patterns:
                    if pattern in logs:
                        # Check if it's actually an error (not just a log message)
                        lines = log_result.stdout.split("\n")
                        error_lines = [line for line in lines if pattern in line.lower() and any(err in line.lower() for err in ["error", "fail", "invalid", "unauthorized"])]
                        if error_lines:
                            license_errors_found = True
                            warn(f"Potential license issue found in {pod_name} logs")
                            print(f"   Pattern: '{pattern}'")
                            print(f"   Sample: {error_lines[0][:100]}...")
                            break
        except Exception as e:
            # Skip pods that can't be logged (may not be ready)
            pass
    
    if not license_errors_found:
        ok("No obvious license-related errors found in pod logs")
    else:
        print("\n💡 If license errors are present, verify:")
        print("   - License key is valid and not expired")
        print("   - Egress to https://beacon.langchain.com is allowed (if not air-gapped)")
        print("   - License secret is correctly mounted in pods")
else:
    warn("Could not retrieve pod names to check logs")


## 2.5. External Services Connectivity

**Important:** Verify that external services (PostgreSQL, Redis, blob storage) are accessible from the cluster. These are critical dependencies for LangSmith.


In [ ]:
from shared._cloud_helpers import (
    get_database_service_name,
    get_cache_service_name,
    get_blob_storage_service_name,
)

# Check external services connectivity
print("### External Services Connectivity Check\n")

# Try to load Terraform outputs to get service endpoints
terraform_outputs_file = artifacts_dir / "terraform-outputs.json"
terraform_outputs = {}

if terraform_outputs_file.exists():
    try:
        with open(terraform_outputs_file) as f:
            terraform_outputs_raw = json.load(f)
        
        # Unwrap Terraform output format
        for key, value in terraform_outputs_raw.items():
            if isinstance(value, dict) and "value" in value:
                terraform_outputs[key] = value["value"]
            else:
                terraform_outputs[key] = value
        
        print("💡 Loaded Terraform outputs for service endpoints\n")
    except Exception as e:
        warn(f"Could not parse Terraform outputs: {e}")
        print("💡 Will attempt basic connectivity checks without endpoint details")
else:
    print("💡 Terraform outputs file not found - will check service connectivity from cluster\n")

# Check PostgreSQL connectivity
print("### PostgreSQL/Database Connectivity\n")
db_service = get_database_service_name()

# Try to find a pod we can exec into for connectivity tests
result = run(
    ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[0].metadata.name}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    test_pod = result.stdout.strip()
    
    # Check if we can reach database (basic connectivity test)
    # This is a simple test - actual connection requires credentials
    db_endpoint = None
    if "rds_endpoint" in terraform_outputs:
        db_endpoint = terraform_outputs["rds_endpoint"]
    elif "postgres_endpoint" in terraform_outputs:
        db_endpoint = terraform_outputs["postgres_endpoint"]
    elif "database_endpoint" in terraform_outputs:
        db_endpoint = terraform_outputs["database_endpoint"]
    
    if db_endpoint:
        # Extract hostname from endpoint (remove port if present)
        db_host = db_endpoint.split(":")[0] if ":" in db_endpoint else db_endpoint
        print(f"Testing connectivity to {db_service} at {db_host}...")
        
        # Try a simple DNS lookup or ping test
        dns_result = run(
            ["kubectl", "exec", "-n", namespace, test_pod, "--", "nslookup", db_host],
            check=False,
            stream=False
        )
        
        if dns_result.returncode == 0:
            ok(f"{db_service} hostname resolves: {db_host}")
        else:
            warn(f"Could not resolve {db_service} hostname")
            print("💡 This may be normal if DNS is not fully configured yet")
    else:
        print(f"💡 {db_service} endpoint not found in Terraform outputs")
        print("   Verify database is accessible from cluster in cloud console")
else:
    print("💡 Could not find pod for connectivity testing")
    print(f"   Manually verify {db_service} is accessible from cluster")

# Check Redis connectivity
print("\n### Redis/Cache Connectivity\n")
cache_service = get_cache_service_name()

if result.returncode == 0 and result.stdout.strip():
    redis_endpoint = None
    if "redis_endpoint" in terraform_outputs:
        redis_endpoint = terraform_outputs["redis_endpoint"]
    elif "cache_endpoint" in terraform_outputs:
        redis_endpoint = terraform_outputs["cache_endpoint"]
    elif "elasticache_endpoint" in terraform_outputs:
        redis_endpoint = terraform_outputs["elasticache_endpoint"]
    
    if redis_endpoint:
        # Extract hostname from endpoint
        redis_host = redis_endpoint.split(":")[0] if ":" in redis_endpoint else redis_endpoint
        print(f"Testing connectivity to {cache_service} at {redis_host}...")
        
        dns_result = run(
            ["kubectl", "exec", "-n", namespace, test_pod, "--", "nslookup", redis_host],
            check=False,
            stream=False
        )
        
        if dns_result.returncode == 0:
            ok(f"{cache_service} hostname resolves: {redis_host}")
        else:
            warn(f"Could not resolve {cache_service} hostname")
            print("💡 This may be normal if DNS is not fully configured yet")
    else:
        print(f"💡 {cache_service} endpoint not found in Terraform outputs")
        print("   Verify cache is accessible from cluster in cloud console")

# Check blob storage (S3/Azure Blob)
print("\n### Blob Storage Connectivity\n")
blob_service = get_blob_storage_service_name()

# Check if blob storage secret exists (indicates it's configured)
blob_secret_result = run(
    ["kubectl", "get", "secret", "-n", namespace, "-o", "jsonpath={.items[*].metadata.name}"],
    check=False,
    stream=False
)

if blob_secret_result.returncode == 0:
    secrets = blob_secret_result.stdout.split()
    blob_secrets = [s for s in secrets if any(keyword in s.lower() for keyword in ["s3", "storage", "blob", "aws"])]
    if blob_secrets:
        ok(f"Blob storage secrets found: {', '.join(blob_secrets)}")
    else:
        print("💡 Blob storage secrets not found (may use IAM roles instead)")

# Check for S3 bucket or blob storage account in Terraform outputs
if "s3_bucket" in terraform_outputs or "bucket_name" in terraform_outputs:
    bucket_name = terraform_outputs.get("s3_bucket") or terraform_outputs.get("bucket_name")
    ok(f"Blob storage bucket/container configured: {bucket_name}")
elif "storage_account" in terraform_outputs:
    storage_account = terraform_outputs["storage_account"]
    ok(f"Azure storage account configured: {storage_account}")
else:
    print(f"💡 Verify {blob_service} is configured and accessible")
    print("   Check Terraform outputs or cloud console for storage resource")

print("\n💡 For comprehensive functional testing of external services,")
print("   see the validation guide for trace submission and attachment tests")


In [ ]:
# Check PVC status
print("### Persistent Volume Claims Status\n")

result = run(
    ["kubectl", "get", "pvc", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
pvc_data = json.loads(result.stdout)

# Display PVCs
print("PVC Details:")
print("=" * 80)
run(["kubectl", "get", "pvc", "-n", namespace, "-o", "wide"], check=True, stream=True)
print("=" * 80)

# Analyze PVC status
bound = 0
pending = 0
total = len(pvc_data.get("items", []))

for pvc in pvc_data.get("items", []):
    status = pvc.get("status", {})
    phase = status.get("phase", "Unknown")
    
    if phase == "Bound":
        bound += 1
    elif phase == "Pending":
        pending += 1
        # Show details for pending PVCs
        name = pvc.get("metadata", {}).get("name", "unknown")
        print(f"\n⚠️  PVC '{name}' is Pending")
        print("   Common causes:")
        print("   - EBS CSI driver not installed")
        print("   - No StorageClass available")
        print("   - Insufficient storage quota")

print(f"\n### PVC Summary")
print(f"Total PVCs: {total}")
print(f"Bound: {bound}")
print(f"Pending: {pending}")

if bound == total and total > 0:
    ok(f"All {total} PVCs are bound")
elif pending > 0:
    warn(f"{pending} PVC(s) still pending - storage issue likely")
    print("💡 Check EBS CSI driver and StorageClasses")
else:
    ok("PVC status looks good")


## 3. Ingress Provisioning Check

**Critical:** The load balancer (ALB for AWS, Application Gateway for Azure) must be provisioned. This is how external traffic reaches LangSmith.

Common issue: Load balancer never appears due to wrong ingress assumptions.


In [ ]:
# Check ingress resources
print("### Ingress Resources\n")

# Get ingress
result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "json"],
    check=False,  # May not exist yet
    stream=False
)

if result.returncode == 0:
    ingress_data = json.loads(result.stdout)
    ingresses = ingress_data.get("items", [])
    
    if ingresses:
        print("Ingress Details:")
        print("=" * 80)
        run(["kubectl", "get", "ingress", "-n", namespace, "-o", "wide"], check=True, stream=True)
        print("=" * 80)
        
        for ingress in ingresses:
            name = ingress.get("metadata", {}).get("name", "unknown")
            status = ingress.get("status", {})
            load_balancer = status.get("loadBalancer", {})
            ingress_hosts = []
            
            # Get ingress hosts
            rules = ingress.get("spec", {}).get("rules", [])
            for rule in rules:
                host = rule.get("host", "")
                if host:
                    ingress_hosts.append(host)
            
            print(f"\nIngress: {name}")
            if ingress_hosts:
                print(f"  Hosts: {', '.join(ingress_hosts)}")
            
            # Check for load balancer address (cloud-agnostic)
            if load_balancer.get("ingress"):
                lb_addresses = [ing.get("hostname", ing.get("ip", "")) for ing in load_balancer["ingress"]]
                if lb_addresses:
                    # Determine load balancer type based on address format
                    lb_type = "Load Balancer"
                    if provider == "aws":
                        if ".elb." in lb_addresses[0] or ".amazonaws.com" in lb_addresses[0]:
                            lb_type = "ALB (Application Load Balancer)"
                    elif provider == "azure":
                        if ".azure.com" in lb_addresses[0] or "appgw" in lb_addresses[0]:
                            lb_type = "Application Gateway"
                    
                    ok(f"{lb_type} provisioned: {', '.join(lb_addresses)}")
                    print(f"  💡 Access LangSmith at: https://{lb_addresses[0]}")
                else:
                    warn("Load balancer ingress entry exists but no address found")
            else:
                warn("Load balancer not yet provisioned (may take a few minutes)")
                print("  💡 Wait a few minutes and check again")
    else:
        warn("No ingress resources found")
        print("💡 Ingress may not be configured in Helm values")
else:
    warn("Could not retrieve ingress resources")
    print("💡 Ingress may not exist yet or namespace is incorrect")

# Check for ingress controller (cloud-agnostic)
print("\n### Ingress Controller\n")

if provider == "aws":
    # Check for ALB Ingress Controller
    result = run(
        ["kubectl", "get", "pods", "-n", "kube-system", "-l", "app.kubernetes.io/name=aws-load-balancer-controller", "-o", "json"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        controller_data = json.loads(result.stdout)
        controllers = controller_data.get("items", [])
        if controllers:
            ok(f"ALB Ingress Controller found ({len(controllers)} pod(s))")
        else:
            warn("ALB Ingress Controller not found")
            print("💡 ALB Ingress Controller must be installed for ingress to work")
    else:
        warn("Could not check ALB Ingress Controller status")

elif provider == "azure":
    # Check for Azure Application Gateway Ingress Controller
    result = run(
        ["kubectl", "get", "pods", "-n", "kube-system", "-l", "app=ingress-azure", "-o", "json"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0:
        controller_data = json.loads(result.stdout)
        controllers = controller_data.get("items", [])
        if controllers:
            ok(f"Azure Application Gateway Ingress Controller found ({len(controllers)} pod(s))")
        else:
            # Also check for AGIC (Application Gateway Ingress Controller)
            result2 = run(
                ["kubectl", "get", "pods", "-n", "kube-system", "-l", "app=ingress-appgw", "-o", "json"],
                check=False,
                stream=False
            )
            if result2.returncode == 0:
                controller_data2 = json.loads(result2.stdout)
                controllers2 = controller_data2.get("items", [])
                if controllers2:
                    ok(f"Application Gateway Ingress Controller found ({len(controllers2)} pod(s))")
                else:
                    warn("Application Gateway Ingress Controller not found")
                    print("💡 Application Gateway Ingress Controller must be installed for ingress to work")
            else:
                warn("Could not check Application Gateway Ingress Controller status")
    else:
        warn("Could not check Application Gateway Ingress Controller status")
else:
    print("💡 Verify ingress controller is installed for your cloud provider")


## 4. Endpoint Reachability Check

Verify that services are accessible and responding. We'll check:
- Service endpoints
- Health check endpoints (if available)
- Internal service connectivity


In [ ]:
# Check services
print("### Service Endpoints\n")

result = run(
    ["kubectl", "get", "svc", "-n", namespace, "-o", "json"],
    check=True,
    stream=False
)
services_data = json.loads(result.stdout)

print("Services:")
print("=" * 80)
run(["kubectl", "get", "svc", "-n", namespace], check=True, stream=True)
print("=" * 80)

services = services_data.get("items", [])
if services:
    ok(f"Found {len(services)} service(s)")
    
    # Check for LoadBalancer services
    lb_services = [svc for svc in services if svc.get("spec", {}).get("type") == "LoadBalancer"]
    if lb_services:
        print(f"\nLoadBalancer services: {len(lb_services)}")
        for svc in lb_services:
            name = svc.get("metadata", {}).get("name", "unknown")
            status = svc.get("status", {}).get("loadBalancer", {})
            if status.get("ingress"):
                lb_address = status["ingress"][0].get("hostname") or status["ingress"][0].get("ip")
                ok(f"Service '{name}' has LoadBalancer: {lb_address}")
            else:
                warn(f"Service '{name}' LoadBalancer pending")
    
    # Test internal connectivity (if we can exec into a pod)
    print("\n### Testing Internal Service Connectivity\n")
    # Try to find a pod we can exec into
    result = run(
        ["kubectl", "get", "pods", "-n", namespace, "-o", "jsonpath={.items[0].metadata.name}"],
        check=False,
        stream=False
    )
    
    if result.returncode == 0 and result.stdout.strip():
        test_pod = result.stdout.strip()
        print(f"Testing connectivity from pod: {test_pod}")
        # Try a simple DNS lookup or curl
        # This is a basic check - actual health endpoints depend on the application
        print("💡 Internal connectivity tests depend on application-specific health endpoints")
else:
    warn("No services found")


## 6. Basic Functional Test (Optional)

**Optional:** Submit a simple test trace to verify the end-to-end pipeline is working. This validates that traces can be ingested, stored, and retrieved.

> **Note:** For comprehensive functional testing (traces, attachments, feedback, datasets), see the full validation guide.


In [ ]:
# Optional: Basic functional test
print("### Basic Functional Test (Optional)\n")

# Check if we have the necessary information to run a test
ingress_result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "jsonpath={.items[0].status.loadBalancer.ingress[0].hostname}"],
    check=False,
    stream=False
)

if ingress_result.returncode == 0 and ingress_result.stdout.strip():
    ingress_host = ingress_result.stdout.strip()
    langsmith_endpoint = f"https://{ingress_host}/api"
    
    print(f"LangSmith endpoint: {langsmith_endpoint}")
    print("\n💡 To run a basic functional test:")
    print("   1. Generate an API key from the LangSmith UI")
    print("   2. Set LANGSMITH_API_KEY environment variable")
    print("   3. Run the test script below (or see validation guide for comprehensive tests)\n")
    
    # Check if API key is available
    api_key = os.environ.get("LANGSMITH_API_KEY", "").strip()
    
    if api_key:
        print("✅ LANGSMITH_API_KEY found - attempting basic trace submission...\n")
        
        try:
            # Simple test: submit a basic trace
            test_code = f'''
import os
import requests
from langsmith import traceable

# Configure LangSmith
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "{api_key}"
os.environ["LANGSMITH_ENDPOINT"] = "{langsmith_endpoint}"
os.environ["LANGSMITH_PROJECT"] = "validation-test"

# Simple traced function
@traceable(name="test_basic_function")
def test_function():
    return "Hello from LangSmith validation test!"

# Run test
try:
    result = test_function()
    print(f"✅ Test trace submitted successfully: {{result}}")
    print(f"💡 Check the LangSmith UI at https://{{ingress_host}} to see the trace")
    print("   Navigate to the 'validation-test' project")
except Exception as e:
    print(f"⚠️  Error submitting trace: {{e}}")
    print("💡 This may be normal if LangSmith is still initializing")
'''
            
            # Try to import langsmith to see if it's available
            try:
                import langsmith
                print("Running basic trace test...")
                exec(test_code)
                ok("Basic functional test completed")
            except ImportError:
                print("⚠️  langsmith package not installed")
                print("💡 Install with: pip install langsmith")
                print("\nTest script (save and run separately):")
                print("=" * 60)
                print(test_code)
                print("=" * 60)
        except Exception as e:
            warn(f"Could not run functional test: {e}")
            print("💡 This is optional - you can test functionality manually in the UI")
    else:
        print("💡 To enable automated testing, set LANGSMITH_API_KEY in your environment")
        print("   Get an API key from: https://{ingress_host}/settings/api-keys")
        print("\nExample test script (run after getting API key):")
        print("=" * 60)
        print(f'''
import os
from langsmith import traceable

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = "<your-api-key>"
os.environ["LANGSMITH_ENDPOINT"] = "{langsmith_endpoint}"
os.environ["LANGSMITH_PROJECT"] = "validation-test"

@traceable(name="test_basic_function")
def test_function():
    return "Hello from LangSmith!"

test_function()
print("Check the UI for the trace!")
''')
        print("=" * 60)
else:
    print("💡 Ingress not available yet - functional test requires accessible endpoint")
    print("   Complete ingress validation first, then return to this section")

print("\n💡 For comprehensive functional testing including:")
print("   - Trace submission & ClickHouse analytics")
print("   - Attachments & blob storage")
print("   - Feedback system")
print("   - Dataset management")
print("   - Agent deployments")
print("   See the full validation guide for detailed test scripts")


## 5. Basic UI Availability Check

**Final validation:** Can we actually access the LangSmith UI through the ingress?

This is the ultimate test - if the UI loads, everything is working.


In [ ]:
import requests
from urllib.parse import urlparse

# Get ingress hostname
print("### UI Availability Check\n")

result = run(
    ["kubectl", "get", "ingress", "-n", namespace, "-o", "jsonpath={.items[0].status.loadBalancer.ingress[0].hostname}"],
    check=False,
    stream=False
)

if result.returncode == 0 and result.stdout.strip():
    ingress_host = result.stdout.strip()
    print(f"Ingress hostname: {ingress_host}")
    
    # Try to access the UI (HTTPS)
    ui_url = f"https://{ingress_host}"
    print(f"\nTesting UI availability at: {ui_url}")
    
    # Cloud-specific messaging
    if provider == "aws":
        print("(This may take a moment if ALB is still provisioning...)\n")
    elif provider == "azure":
        print("(This may take a moment if Application Gateway is still provisioning...)\n")
    else:
        print("(This may take a moment if load balancer is still provisioning...)\n")
    
    try:
        # Use a short timeout and allow redirects
        response = requests.get(ui_url, timeout=10, allow_redirects=True, verify=False)
        if response.status_code == 200:
            ok(f"UI is accessible! Status: {response.status_code}")
            print(f"💡 Open in browser: {ui_url}")
        elif response.status_code in [301, 302, 307, 308]:
            ok(f"UI redirects (status: {response.status_code}) - likely working")
            print(f"💡 Redirect location: {response.headers.get('Location', 'N/A')}")
            print(f"💡 Open in browser: {ui_url}")
        else:
            warn(f"UI returned status {response.status_code}")
            print("💡 UI may still be starting or there may be a configuration issue")
    except requests.exceptions.SSLError:
        # SSL errors might be expected if using self-signed certs
        warn("SSL verification failed (may be expected with self-signed certs)")
        print(f"💡 Try accessing: {ui_url}")
        print("   Browser may show security warning - this is normal for self-signed certs")
    except requests.exceptions.Timeout:
        warn("UI request timed out")
        if provider == "aws":
            print("💡 ALB may still be provisioning, or ingress is not fully configured")
            print(f"   Check AWS console for ALB status, then try: {ui_url}")
        elif provider == "azure":
            print("💡 Application Gateway may still be provisioning, or ingress is not fully configured")
            print(f"   Check Azure portal for Application Gateway status, then try: {ui_url}")
        else:
            print(f"   Try again in a few minutes: {ui_url}")
    except requests.exceptions.ConnectionError as e:
        warn(f"Could not connect to UI: {e}")
        if provider == "aws":
            print("💡 ALB may still be provisioning")
            print(f"   Check AWS console for ALB status, then try: {ui_url}")
        elif provider == "azure":
            print("💡 Application Gateway may still be provisioning")
            print(f"   Check Azure portal for Application Gateway status, then try: {ui_url}")
        else:
            print(f"   Try again in a few minutes: {ui_url}")
    except Exception as e:
        warn(f"Error accessing UI: {e}")
        print(f"💡 Manual check: Open {ui_url} in a browser")
else:
    warn("Could not determine ingress hostname")
    print("💡 Ingress may not be provisioned yet")
    if provider == "aws":
        print("   Run the ingress check above and wait for ALB to be created")
    elif provider == "azure":
        print("   Run the ingress check above and wait for Application Gateway to be created")
    else:
        print("   Run the ingress check above and wait for load balancer to be created")


## Collecting Diagnostic Artifacts

Save cluster state snapshots for future troubleshooting reference.


In [ ]:
from datetime import datetime

# Create diagnostic snapshot
print("### Collecting Diagnostic Artifacts\n")

timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
diagnostics_dir = artifacts_dir / f"diagnostics-{timestamp}"
diagnostics_dir.mkdir(exist_ok=True)

print(f"Saving diagnostics to: {diagnostics_dir}\n")

# Save various cluster states
diagnostics = [
    ("pods", ["kubectl", "get", "pods", "-n", namespace, "-o", "yaml"]),
    ("services", ["kubectl", "get", "svc", "-n", namespace, "-o", "yaml"]),
    ("ingress", ["kubectl", "get", "ingress", "-n", namespace, "-o", "yaml"]),
    ("pvc", ["kubectl", "get", "pvc", "-n", namespace, "-o", "yaml"]),
    ("deployments", ["kubectl", "get", "deployments", "-n", namespace, "-o", "yaml"]),
    ("events", ["kubectl", "get", "events", "-n", namespace, "--sort-by=.lastTimestamp"]),
]

for name, cmd in diagnostics:
    try:
        result = run(cmd, check=False, stream=False)
        output_file = diagnostics_dir / f"{name}.txt"
        with open(output_file, "w") as f:
            f.write(result.stdout)
            if result.stderr:
                f.write("\n\nSTDERR:\n")
                f.write(result.stderr)
        print(f"✅ Saved {name}")
    except Exception as e:
        print(f"⚠️  Could not save {name}: {e}")

ok(f"Diagnostics saved to: {diagnostics_dir}")
print("\n💡 These artifacts can be used for troubleshooting or support tickets")


## Go/No-Go Checklist

Review this checklist. All items should be ✅ before considering the deployment ready.

### ✅ Validation Checklist

- [ ] All pods are running and ready
- [ ] License key is properly configured (no errors in logs)
- [ ] All PVCs are bound
- [ ] External services are accessible (PostgreSQL, Redis, blob storage)
- [ ] Ingress/load balancer is provisioned
- [ ] Services are accessible
- [ ] UI is reachable (or load balancer is provisioning)
- [ ] Basic functional test passed (optional)
- [ ] Diagnostic artifacts collected

### 🎯 Next Steps

**If all checks pass:**
- ✅ You have a working baseline deployment
- ✅ You're on a supported path
- ✅ Ready to proceed to Module 2 (SSO/OIDC configuration)
- 💡 For comprehensive functional testing, see the full validation guide

**If checks fail:**
- Review the warnings above
- Check diagnostic artifacts
- Common issues:
  - **PVCs pending:** Storage CSI driver not installed
- **Load balancer not appearing:** Wrong ingress configuration
- **Pods not ready:** Check events and logs
- **UI not accessible:** Wait for load balancer provisioning (can take 5-10 minutes)
- **License errors:** Verify license key is valid and secret is correctly mounted
- **External services unreachable:** Check network connectivity and security groups

### 📋 Baseline Reference

This validation checklist becomes your **baseline reference** for future troubleshooting. Save the diagnostic artifacts and refer back to this state when investigating issues.
